# V34 — noise-adaptive residual Demucs

This notebook converts the current best Demucs into one always-on,
noise-adaptive subtraction model. A continuous spectral conditioner modulates
the Demucs bottleneck with zero-initialized FiLM parameters. The enhanced
signal is always computed as `input - predicted_noise`; no category switch,
threshold, filename rule, correlation lookup, or natural/synthetic routing is
used at inference.

The notebook reuses existing TTS/pseudo-pair caches and never reruns Parler,
BEATs or WavLM. It starts exactly from the selected V33 checkpoint when
available, otherwise from V32 epoch 6. All V34 outputs are written separately.


## 0 — One-cell startup preflight


In [ ]:
# Run first in a fresh Colab GPU runtime. No TTS/BEATs/WavLM dependency is loaded.
import os, sys, json, subprocess, importlib.util, importlib.metadata
from pathlib import Path

os.environ["USE_TF"]="0";os.environ["USE_FLAX"]="0"
try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():drive.mount("/content/drive")
except ImportError as exc:raise RuntimeError("This notebook is designed for Google Colab.") from exc

required={"numpy":"numpy","pandas":"pandas","soundfile":"soundfile","librosa":"librosa",
          "scipy":"scipy","tqdm":"tqdm","huggingface_hub":"huggingface_hub"}
missing=[pkg for module,pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:subprocess.check_call([sys.executable,"-m","pip","install","-q",*missing])
try:tv=importlib.metadata.version("transformers")
except importlib.metadata.PackageNotFoundError:tv=None
try:pv=importlib.metadata.version("protobuf")
except importlib.metadata.PackageNotFoundError:pv=None
if tv!="4.46.1" or pv!="6.31.1" or importlib.util.find_spec("denoiser") is None:
    if "transformers" in sys.modules or "google.protobuf" in sys.modules:
        raise RuntimeError("Restart once: transformers/protobuf was imported before this startup cell.")
    subprocess.check_call([sys.executable,"-m","pip","install","-q","--upgrade",
                           "transformers==4.46.1","tokenizers==0.20.3","protobuf==6.31.1"])
    if importlib.util.find_spec("denoiser") is None:
        subprocess.check_call([sys.executable,"-m","pip","install","-q","--no-deps","denoiser==0.1.5"])

import torch
if not torch.cuda.is_available():raise RuntimeError("Select a Colab GPU runtime before continuing.")
from google.protobuf import runtime_version as _protobuf_check
from transformers import AutoModel as _transformers_check
from denoiser import pretrained as _demucs_check
_m=_demucs_check.dns64(pretrained=False)
assert _m.sample_rate==16000 and _m.chin==1
del _m,_protobuf_check,_transformers_check,_demucs_check

root=Path("/content/drive/MyDrive/PHD/Datathon")
v32=root/"track2_phase2_v32_indicparler_beats_demucs"/"run_v1"
v33=root/"track2_phase2_v33_multievent_demucs"/"run_v1"
pair_candidates=[v33/"pseudo_pairs_multievent.jsonl",v32/"pseudo_pairs.jsonl"]
pin_candidates=[Path("/content/drive/MyDrive/Datathon/track2_v6_direct_reconstruction/sravaani_revision_pin.json"),
                root/"track2_v6_direct_reconstruction"/"sravaani_revision_pin.json"]
required_paths=[root/"validation"/"validationMetadata.json",root/"input_data.zip",root/"transcripts.jsonl",
                v32/"demucs_epoch_06.pt"]
if not any(p.is_file() for p in pair_candidates):required_paths.append(pair_candidates[0])
if not any(p.is_file() for p in pin_candidates):required_paths.append(pin_candidates[-1])
missing_paths=[str(p) for p in required_paths if not p.exists()]
if missing_paths:raise FileNotFoundError("Startup preflight missing:\n"+"\n".join(missing_paths))
print("STARTUP PREFLIGHT PASS | CUDA",torch.cuda.get_device_name(0),"| transformers",tv,
      "| protobuf",pv,"| pseudo manifest",next(p for p in pair_candidates if p.is_file()))


Mounted at /content/drive
STARTUP PREFLIGHT PASS | CUDA Tesla T4 | transformers 5.18.0 | protobuf 6.33.6 | pseudo manifest /content/drive/MyDrive/PHD/Datathon/track2_phase2_v33_multievent_demucs/run_v1/pseudo_pairs_multievent.jsonl


## 1 — Configuration and exact audio/submission contract


In [ ]:
from pathlib import Path
import os,io,re,gc,json,math,wave,shutil,random,hashlib,zipfile,unicodedata,time
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from tqdm.auto import tqdm
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from huggingface_hub import hf_hub_download,snapshot_download
from IPython.display import display,Audio

ROOT=Path("/content/drive/MyDrive/PHD/Datathon")
VAL_ROOT=ROOT/"validation";META_PATH=VAL_ROOT/"validationMetadata.json"
SOURCE_AUDIO=ROOT/"input_data.zip";TRANSCRIPTS_JSONL=ROOT/"transcripts.jsonl"
V32_RUN=ROOT/"track2_phase2_v32_indicparler_beats_demucs"/"run_v1"
V33_RUN=ROOT/"track2_phase2_v33_multievent_demucs"/"run_v1"
OUT=ROOT/"track2_phase2_v34_noise_adaptive_residual_demucs"
LOCAL=Path("/content/track2_phase2_v34_noise_adaptive_residual_demucs")
RUN=OUT/"run_v1";WORK=LOCAL/"run_v1"
for p in (OUT,LOCAL,RUN,WORK):p.mkdir(parents=True,exist_ok=True)

PSEUDO_MANIFEST=(V33_RUN/"pseudo_pairs_multievent.jsonl" if
    (V33_RUN/"pseudo_pairs_multievent.jsonl").is_file() else V32_RUN/"pseudo_pairs.jsonl")
BASELINE_CHECKPOINT=V32_RUN/"demucs_epoch_06.pt"
v33_selection=V33_RUN/"selected_model.json"
if v33_selection.is_file():
    candidate=Path(json.loads(v33_selection.read_text()).get("checkpoint",""))
    if candidate.is_file():BASELINE_CHECKPOINT=candidate

SR=16000;SEED=517
NATURAL_DEV_LIMIT=100;NATURAL_MONITOR_LIMIT=100
SEGMENT=51200;BATCH_SIZE=4;FINETUNE_EPOCHS=3
LR_ADAPTER=1e-4;LR_DECODER=2e-6;LR_FULL=1e-6
IDENTITY_PROBABILITY=.05;RANDOM_CROP_PROBABILITY=.15;EVENT_CENTRED_PROBABILITY=.80
NOISE_LOSS_WEIGHT=.30;STFT_WEIGHT=.05;WAVE_WEIGHT=.10;PRESERVE_WEIGHT=.10
CHUNK_SAMPLES=51200;CHUNK_HOP=25600;EXPECTED_PHASE2_CLIPS=3178
RUN_PHASE2=True
CONFIRM_RULES_ALLOW_SOURCE_DERIVED_TRAINING=True

ASR_REPO="ARTPARK-IISc/SraVaani-1.0"
ASR_PIN_CANDIDATES=[Path("/content/drive/MyDrive/Datathon/track2_v6_direct_reconstruction/sravaani_revision_pin.json"),
                    ROOT/"track2_v6_direct_reconstruction"/"sravaani_revision_pin.json"]
ASR_PIN=next((p for p in ASR_PIN_CANDIDATES if p.is_file()),ASR_PIN_CANDIDATES[-1])
DEVICE=torch.device("cuda");IMPLEMENTATION="v34_noise_adaptive_residual_demucs_v1"
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
print("V34 run:",RUN);print("Baseline:",BASELINE_CHECKPOINT);print("Pseudo manifest:",PSEUDO_MANIFEST)


V34 run: /content/drive/MyDrive/PHD/Datathon/track2_phase2_v34_noise_adaptive_residual_demucs/run_v1
Baseline: /content/drive/MyDrive/PHD/Datathon/track2_phase2_v32_indicparler_beats_demucs/run_v1/demucs_epoch_06.pt
Pseudo manifest: /content/drive/MyDrive/PHD/Datathon/track2_phase2_v33_multievent_demucs/run_v1/pseudo_pairs_multievent.jsonl


In [ ]:
def jsonable(x):
    if isinstance(x, dict): return {str(k):jsonable(v) for k,v in x.items()}
    if isinstance(x, (list, tuple)): return [jsonable(v) for v in x]
    if isinstance(x, Path): return str(x)
    if isinstance(x, np.ndarray): return jsonable(x.tolist())
    if isinstance(x, np.generic): return jsonable(x.item())
    if isinstance(x, float) and not math.isfinite(x): return None
    return x

def put_json(path, obj):
    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
    t=p.with_name(p.name+'.tmp')
    with open(t,'w',encoding='utf-8') as f:
        json.dump(jsonable(obj),f,indent=2,ensure_ascii=False,allow_nan=False)
    os.replace(t,p)

def sha_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(4<<20),b''): h.update(b)
    return h.hexdigest()

def sha_json(obj):
    return hashlib.sha256(json.dumps(jsonable(obj),sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()

def canonical_pcm(y):
    a=np.asarray(y,np.float64)
    if a.ndim!=1 or not len(a) or not np.isfinite(a).all():
        raise ValueError('Wave must be finite, nonempty, and mono.')
    # Deterministic rounding, not peak normalization. Used for metrics AND export.
    return np.clip(np.rint(a*32768.0),-32768,32767).astype('<i2')

def pcm_roundtrip(y): return canonical_pcm(y).astype(np.float64)/32768.0

def wav_bytes(y, sr=16000):
    buf=io.BytesIO()
    with wave.open(buf,'wb') as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr)
        w.writeframes(canonical_pcm(y).tobytes())
    return buf.getvalue()

def write_pcm(path,y,sr=16000):
    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
    data=wav_bytes(y,sr); tmp=p.with_name(p.name+'.tmp')
    with open(tmp,'wb') as f: f.write(data)
    os.replace(tmp,p)
    return hashlib.sha256(data).hexdigest()

def read_wave_strict(path_or_bytes,sr=16000):
    source=io.BytesIO(path_or_bytes) if isinstance(path_or_bytes,(bytes,bytearray)) else path_or_bytes
    y,fs=sf.read(source,dtype='float32',always_2d=True)
    if fs!=sr or y.shape[1]!=1:
        raise ValueError(f'Expected native {sr}-Hz mono audio, got {fs} Hz / {y.shape[1]} channels. No unvalidated resampling.')
    a=y[:,0]
    if not len(a) or not np.isfinite(a).all(): raise ValueError('Empty or invalid audio.')
    return a

def wav_audit(path_or_bytes,expected_frames,sr=16000):
    data=path_or_bytes if isinstance(path_or_bytes,bytes) else Path(path_or_bytes).read_bytes()
    with wave.open(io.BytesIO(data),'rb') as w:
        got=(w.getframerate(),w.getnchannels(),w.getsampwidth(),w.getnframes(),w.getcomptype())
        wanted=(sr,1,2,int(expected_frames),'NONE')
        if got!=wanted: raise ValueError(f'WAV compliance error: {got} != {wanted}')
        pcm=w.readframes(expected_frames)
        if len(pcm)!=expected_frames*2: raise ValueError('Truncated PCM payload.')
    return hashlib.sha256(data).hexdigest()

class StrictWaveSource:
    """One explicit directory/ZIP, exact IDs; no duplicate stems or inferred paths."""
    def __init__(self,path,required_ids):
        self.path=str(path); self.archive=None; self.index={}
        required_ids=set(map(str,required_ids))
        p=Path(path)
        if p.is_file() and zipfile.is_zipfile(p):
            self.archive=zipfile.ZipFile(p)
            names=[z.filename for z in self.archive.infolist() if not z.is_dir() and z.filename.lower().endswith('.wav') and '__MACOSX' not in z.filename]
        elif p.is_dir(): names=[str(q) for q in p.rglob('*') if q.is_file() and q.suffix.lower()=='.wav']
        else: raise FileNotFoundError(p)
        for name in names:
            cid=Path(name).stem
            if cid not in required_ids: continue
            if cid in self.index: raise ValueError(f'Duplicate waveform ID: {cid} in {p}')
            self.index[cid]=name
        missing=required_ids-set(self.index)
        if missing: raise ValueError(f'{p}: {len(missing)} missing IDs; examples {sorted(missing)[:3]}')
    def bytes(self,cid):
        loc=self.index[cid]
        return self.archive.read(loc) if self.archive is not None else Path(loc).read_bytes()
    def load(self,cid): return read_wave_strict(self.bytes(cid))
    def close(self):
        if self.archive is not None: self.archive.close()

def si_sdr(estimate,reference):
    """Zero-mean SI-SDR, no alignment/resampling; 120-dB numerical ceiling only."""
    x=np.asarray(estimate,np.float64); s=np.asarray(reference,np.float64)
    if x.shape!=s.shape or x.ndim!=1: raise ValueError('SI-SDR length/shape mismatch.')
    if not np.isfinite(x).all() or not np.isfinite(s).all(): raise ValueError('Nonfinite SI-SDR input.')
    x=x-x.mean(); s=s-s.mean(); ss=float(s@s)
    if ss<=1e-18: raise ValueError('Silent reference: SI-SDR undefined, not silently excluded.')
    target=(float(x@s)/ss)*s; e=x-target
    num=float(target@target); den=float(e@e)
    value=10*math.log10(max(num,ss*1e-16)/max(den,ss*1e-12))
    return float(min(120.,value))

def hyp_text(h):
    if isinstance(h,str): return h.strip()
    if isinstance(h,dict) and isinstance(h.get('text'),str): return h['text'].strip()
    if hasattr(h,'text') and isinstance(h.text,str): return h.text.strip()
    if isinstance(h,(list,tuple)) and len(h)==1: return hyp_text(h[0])
    raise TypeError(f'Unrecognized ASR hypothesis type: {type(h)}. Refusing an invented empty transcript.')

def pack_submission(paths,asr,expected_frames,output_zip,sr=16000):
    """Only final audio and matching ASR enter the ZIP; diagnostics/clean refs never do."""
    ids=sorted(paths)
    if set(ids)!=set(asr) or set(ids)!=set(expected_frames): raise ValueError('Submission ID mismatch.')
    for cid in ids:
        if not cid or Path(cid).name!=cid or any(c in cid for c in ('/','\\','\n','\r')): raise ValueError('Unsafe clip ID.')
        h=wav_audit(paths[cid],expected_frames[cid],sr)
        if h!=asr[cid]['wav_sha256']: raise ValueError(f'ASR/audio hash mismatch: {cid}')
        if not isinstance(asr[cid]['text'],str): raise ValueError('ASR text must be a string.')
    p=Path(output_zip); p.parent.mkdir(parents=True,exist_ok=True); tmp=p.with_name(p.name+'.tmp')
    tr=''.join(json.dumps(dict(clip_id=c,text=asr[c]['text']),ensure_ascii=False,allow_nan=False)+'\n' for c in ids).encode('utf-8')
    with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED,compresslevel=9,allowZip64=True) as z:
        z.writestr('transcripts.jsonl',tr)
        for cid in ids: z.write(paths[cid],cid+'.wav')
    with zipfile.ZipFile(tmp) as z:
        wanted={'transcripts.jsonl'}|{c+'.wav' for c in ids}
        if len(z.namelist())!=len(wanted) or set(z.namelist())!=wanted: raise ValueError('Archive contents mismatch.')
        if z.testzip() is not None: raise ValueError('Archive CRC failure.')
        for cid in ids:
            data=z.read(cid+'.wav')
            if wav_audit(data,expected_frames[cid],sr)!=asr[cid]['wav_sha256']: raise ValueError('Packaged audio mismatch.')
        rows=[json.loads(x) for x in z.read('transcripts.jsonl').decode().splitlines()]
        if len(rows)!=len(ids) or {r['clip_id'] for r in rows}!=set(ids): raise ValueError('Packaged transcript mismatch.')
    os.replace(tmp,p)
    return dict(zip_path=str(p),zip_sha256=sha_file(p),clips=len(ids),transcripts_sha256=hashlib.sha256(tr).hexdigest())


## 2 — Reconstruct the fixed real validation splits


In [ ]:
def wav_index(root):
    if not root.is_dir(): raise FileNotFoundError(root)
    result={}
    for p in sorted(root.rglob('*')):
        if p.is_file() and p.suffix.lower()=='.wav':
            if p.name in result: raise ValueError(f'Duplicate filename in {root}: {p.name}')
            result[p.name]=p
    return result

def filename_group(name):
    # Dataset filenames end the apparent participant/session token with digits,
    # followed by a numeric utterance token. Retain prefix before that token.
    m=re.match(r'^(.*?\d{6,})_\d+_',Path(name).stem)
    return m.group(1) if m else Path(name).stem

def make_validation_tables(meta_path,root):
    records=json.loads(meta_path.read_text(encoding='utf-8'))
    if not isinstance(records,list): raise ValueError('Expected a JSON list')
    names=[r['segmentFileName'] for r in records]
    if len(names)!=len(set(names)): raise ValueError('Duplicate metadata filenames')
    noisy=wav_index(root/'syntheticNoiseAudio')
    clean=wav_index(root/'syntheticCleanRefAudio')
    natural=wav_index(root/'naturalNoisyAudio')
    pairs=[]; nat=[]
    for r in tqdm(records,desc='Audit validation audio'):
        name=r['segmentFileName']; cid=Path(name).stem
        if Path(name).name!=name: raise ValueError('Expected basename in segmentFileName')
        if not isinstance(r['syntheticData'],bool): raise ValueError('syntheticData must be Boolean')
        common=dict(clip_id=cid,transcript=r.get('transcript',''),language=r.get('language',''),
                    events=r.get('NoiseSubCategoryTimeStamp',[]))
        if r['syntheticData']:
            if name not in noisy or name not in clean: raise FileNotFoundError(f'Missing pair: {name}')
            x=read_wave_strict(noisy[name]); s=read_wave_strict(clean[name])
            if x.shape!=s.shape: raise ValueError(f'Pair length mismatch: {name}')
            if np.array_equal(x,s): raise ValueError(f'Identical noisy/clean pair: {name}')
            if np.mean(s.astype(np.float64)**2)<1e-10: raise ValueError(f'Silent clean target: {name}')
            pairs.append(dict(**common,noisy_path=str(noisy[name]),clean_path=str(clean[name]),
                group_proxy=filename_group(name),frames=len(x),
                noisy_hash=hashlib.sha256(x.tobytes()).hexdigest(),
                clean_hash=hashlib.sha256(s.tobytes()).hexdigest()))
        else:
            if name not in natural: raise FileNotFoundError(f'Missing natural WAV: {name}')
            x=read_wave_strict(natural[name])
            nat.append(dict(**common,noisy_path=str(natural[name]),frames=len(x),
                            noisy_hash=hashlib.sha256(x.tobytes()).hexdigest()))
    # Union grouping proxies and exact clean/noisy duplicates, including cross roles.
    parent=list(range(len(pairs)))
    def find(i):
        while parent[i]!=i:
            parent[i]=parent[parent[i]]; i=parent[i]
        return i
    def union(a,b): parent[find(a)]=find(b)
    seen={}
    for i,r in enumerate(pairs):
        for key in [('proxy',r['group_proxy']),('audio',r['clean_hash']),('audio',r['noisy_hash'])]:
            if key in seen: union(i,seen[key])
            else: seen[key]=i
    groups={}
    for i in range(len(pairs)): groups.setdefault(find(i),[]).append(i)
    if len(groups)<10: raise ValueError('Too few independent groups for a useful split')
    ordered=sorted(groups.values(),key=lambda ids: hashlib.sha256(
        (str(SEED)+'|'+min(pairs[i]['clip_id'] for i in ids)).encode()).hexdigest())
    ng=len(ordered); n_hold=max(1,round(.15*ng)); n_dev=max(1,round(.15*ng))
    for j,ids in enumerate(ordered):
        split='holdout' if j<n_hold else 'dev' if j<n_hold+n_dev else 'train'
        group=min(pairs[i]['clip_id'] for i in ids)
        for i in ids: pairs[i].update(split=split,split_group=group)
    return pd.DataFrame(pairs),pd.DataFrame(nat)

PAIRS,NATURAL=make_validation_tables(META_PATH,VAL_ROOT)
if len(PAIRS)!=695 or len(NATURAL)!=1806:
    raise ValueError(f'Metadata changed: {len(PAIRS)} paired / {len(NATURAL)} natural. Inspect before proceeding.')
DATA_SHA=sha_json(dict(metadata=sha_file(META_PATH),audio=PAIRS[
    ['clip_id','noisy_hash','clean_hash','split']].to_dict('records')))

def natural_sample(df,exclude_ids=(),limit=None):
    rows=df[~df.clip_id.isin(exclude_ids)].copy()
    rows['_key']=rows.clip_id.map(lambda s:hashlib.sha256((str(SEED)+s).encode()).hexdigest())
    rows=rows.sort_values('_key')
    return rows if limit is None else rows.head(limit)
# Prior V27 natural sample is explicitly the development set, not a new holdout.

NAT_DEV=natural_sample(NATURAL,limit=NATURAL_DEV_LIMIT)
NAT_MON=natural_sample(NATURAL,exclude_ids=set(NAT_DEV.clip_id),limit=NATURAL_MONITOR_LIMIT)
# V34 uses the original holdout as the untouched genuine selection set and
# adds the former development split to training. No holdout pair is fitted.
SYN_DEV=PAIRS[PAIRS.split=="holdout"].copy()
SYN_MON=PAIRS[PAIRS.split=="dev"].copy()      # fit diagnostic only; never selects a checkpoint
TRAIN_ROWS=PAIRS[PAIRS.split!="holdout"].copy()
assert set(TRAIN_ROWS.clip_id).isdisjoint(set(SYN_DEV.clip_id))
assert set(NAT_DEV.clip_id).isdisjoint(NAT_MON.clip_id)
print("Genuine training / untouched synthetic selection:",len(TRAIN_ROWS),len(SYN_DEV))
print("Natural development / monitoring:",len(NAT_DEV),len(NAT_MON))


Audit validation audio:   0%|          | 0/2501 [00:00<?, ?it/s]

Genuine training / untouched synthetic selection: 591 104
Natural development / monitoring: 100 100


## 3 — Restore completed pseudo pairs; no regeneration


In [ ]:
def event_dicts(raw):
    if raw is None:return []
    if isinstance(raw,str):
        raw=raw.strip()
        if not raw:return []
        raw=json.loads(raw)
    if isinstance(raw,np.ndarray):raw=raw.tolist()
    out=[]
    for e in raw if isinstance(raw,(list,tuple)) else []:
        if not isinstance(e,dict):continue
        start=e.get("start",e.get("onset"));end=e.get("end",e.get("offset"))
        if start is None or end is None:continue
        out.append(dict(start=float(start),end=float(end),category=str(e.get("category","unknown")),
                        tag=str(e.get("tag","unknown")),profile_id=str(e.get("profile_id",""))))
    return out

def clean_text(text):
    text=re.sub(r"<[^>]*>|\[[^\]]*\]"," ",str(text or ""))
    return re.sub(r"\s+"," ",unicodedata.normalize("NFC",text)).strip()

records=[json.loads(line) for line in PSEUDO_MANIFEST.read_text(encoding="utf-8").splitlines() if line.strip()]
PSEUDO=pd.DataFrame(records)
required={"pair_id","source_clip","split","noisy_path","clean_path","frames","events"}
if required-set(PSEUDO.columns):raise ValueError(f"Pseudo manifest misses {sorted(required-set(PSEUDO.columns))}")
if len(PSEUDO)!=5000 or PSEUDO.pair_id.duplicated().any():raise ValueError("Expected 5,000 unique pseudo pairs")
bad=[]
for r in tqdm(PSEUDO.to_dict("records"),desc="Validate cached pseudo-pair paths"):
    if not Path(r["noisy_path"]).is_file() or not Path(r["clean_path"]).is_file():bad.append(r["pair_id"])
    if not event_dicts(r["events"]):bad.append(r["pair_id"]+":no_event")
if bad:raise RuntimeError(f"Invalid pseudo cache ({len(bad)}): {bad[:5]}")
TRANSCRIPTS=[json.loads(line) for line in TRANSCRIPTS_JSONL.read_text(encoding="utf-8").splitlines() if line.strip()]
TRANSCRIPT_MAP={str(r["clip_id"]):clean_text(r["text"]) for r in TRANSCRIPTS}
if len(TRANSCRIPT_MAP)!=EXPECTED_PHASE2_CLIPS:raise ValueError("Unexpected transcript count")
event_counts=PSEUDO.events.map(lambda z:len(event_dicts(z)))
print("PSEUDO RESTORE PASS |",len(PSEUDO),"pairs |",PSEUDO.source_clip.nunique(),"source clips",
      "| mean events",float(event_counts.mean()),"| manifest",PSEUDO_MANIFEST)


Validate cached pseudo-pair paths:   0%|          | 0/5000 [00:00<?, ?it/s]

PSEUDO RESTORE PASS | 5000 pairs | 3178 source clips | mean events 1.794 | manifest /content/drive/MyDrive/PHD/Datathon/track2_phase2_v33_multievent_demucs/run_v1/pseudo_pairs_multievent.jsonl


## 4 — Zero-initialized noise-adaptive residual Demucs


In [ ]:
from denoiser import pretrained as demucs_pretrained
from denoiser.resample import upsample2,downsample2

base=demucs_pretrained.dns64(pretrained=False)
if base.sample_rate!=SR or base.chin!=1:raise RuntimeError("Unexpected Demucs configuration")
base.load_state_dict(torch.load(BASELINE_CHECKPOINT,map_location="cpu",weights_only=True),strict=True)
BASELINE_SHA=sha_file(BASELINE_CHECKPOINT)

class SpectralNoiseConditioner(nn.Module):
    def __init__(self,n_fft=512,hop=160,embedding=128):
        super().__init__();self.n_fft=n_fft;self.hop=hop
        self.register_buffer("window",torch.hann_window(n_fft),persistent=False)
        features=2*(n_fft//2+1)
        self.net=nn.Sequential(nn.LayerNorm(features),nn.Linear(features,256),nn.SiLU(),
                               nn.Linear(256,embedding),nn.SiLU())
    def forward(self,wave):
        with torch.amp.autocast(device_type=wave.device.type,enabled=False):
            spec=torch.stft(wave.float(),self.n_fft,hop_length=self.hop,window=self.window.float(),
                            return_complex=True,pad_mode="constant").abs()
            feat=torch.log1p(10*spec);feat=torch.cat([feat.mean(-1),feat.std(-1)],-1)
        return self.net(feat)

class NoiseAdaptiveResidualDemucs(nn.Module):
    """Exact Demucs forward plus continuous FiLM and learned residual strength."""
    def __init__(self,demucs,embedding=128,film_scale=.10,strength_range=.15):
        super().__init__();self.base=demucs;self.conditioner=SpectralNoiseConditioner(embedding=embedding)
        bottleneck=int(self.base.encoder[-1][0].out_channels)
        self.film=nn.Linear(embedding,2*bottleneck)
        self.strength=nn.Linear(embedding,1)
        nn.init.zeros_(self.film.weight);nn.init.zeros_(self.film.bias)
        nn.init.zeros_(self.strength.weight);nn.init.zeros_(self.strength.bias)
        self.film_scale=float(film_scale);self.strength_range=float(strength_range)
    def forward(self,mix,return_aux=False):
        if mix.dim()==2:mix=mix.unsqueeze(1)
        original=mix;embedding=self.conditioner(original[:,0])
        if self.base.normalize:
            std=original.mean(1,keepdim=True).std(-1,keepdim=True);x=original/(self.base.floor+std)
        else:std=1.;x=original
        length=x.shape[-1];x=F.pad(x,(0,self.base.valid_length(length)-length))
        if self.base.resample==2:x=upsample2(x)
        elif self.base.resample==4:x=upsample2(upsample2(x))
        skips=[]
        for encode in self.base.encoder:x=encode(x);skips.append(x)
        x=x.permute(2,0,1);x,_=self.base.lstm(x);x=x.permute(1,2,0)
        gamma,beta=self.film(embedding).chunk(2,-1)
        x=x*(1+self.film_scale*torch.tanh(gamma)[:,:,None])+self.film_scale*torch.tanh(beta)[:,:,None]
        for decode in self.base.decoder:
            skip=skips.pop(-1);x=decode(x+skip[...,:x.shape[-1]])
        if self.base.resample==2:x=downsample2(x)
        elif self.base.resample==4:x=downsample2(downsample2(x))
        speech_candidate=std*x[...,:length]
        strength=1+self.strength_range*torch.tanh(self.strength(embedding))[:,:,None]
        predicted_noise=strength*(original-speech_candidate)
        enhanced=original-predicted_noise
        return (enhanced,predicted_noise,strength,embedding) if return_aux else enhanced

model=NoiseAdaptiveResidualDemucs(base)
with torch.no_grad():
    probe=torch.randn(2,1,16000)
    expected=base(probe);actual=model(probe)
    rel=float((expected-actual).norm()/expected.norm().clamp_min(1e-8))
if rel>1e-5:raise RuntimeError(f"Zero-initialized wrapper does not reproduce baseline: {rel}")
INITIAL_CHECKPOINT=RUN/"adaptive_initial.pt"
if not INITIAL_CHECKPOINT.exists():torch.save({k:v.detach().cpu() for k,v in model.state_dict().items()},INITIAL_CHECKPOINT)
print("ADAPTIVE INITIALIZATION PASS | relative baseline error",rel,"| baseline SHA",BASELINE_SHA)


RuntimeError: Zero-initialized wrapper does not reproduce baseline: 2.3580925699207e-05

In [ ]:
from denoiser import pretrained as demucs_pretrained
from denoiser.resample import upsample2, downsample2

base = demucs_pretrained.dns64(pretrained=False)
if base.sample_rate != SR or base.chin != 1:
    raise RuntimeError("Unexpected Demucs configuration")

base.load_state_dict(
    torch.load(BASELINE_CHECKPOINT, map_location="cpu", weights_only=True),
    strict=True
)
BASELINE_SHA = sha_file(BASELINE_CHECKPOINT)


class SpectralNoiseConditioner(nn.Module):
    def __init__(self, n_fft=512, hop=160, embedding=128):
        super().__init__()
        self.n_fft = n_fft
        self.hop = hop

        self.register_buffer(
            "window",
            torch.hann_window(n_fft),
            persistent=False
        )

        features = 2 * (n_fft // 2 + 1)

        self.net = nn.Sequential(
            nn.LayerNorm(features),
            nn.Linear(features, 256),
            nn.SiLU(),
            nn.Linear(256, embedding),
            nn.SiLU(),
        )

    def forward(self, wave):
        with torch.amp.autocast(
            device_type=wave.device.type,
            enabled=False
        ):
            spec = torch.stft(
                wave.float(),
                self.n_fft,
                hop_length=self.hop,
                window=self.window.float(),
                return_complex=True,
                pad_mode="constant",
            ).abs()

            feat = torch.log1p(10 * spec)
            feat = torch.cat(
                [feat.mean(-1), feat.std(-1)],
                dim=-1
            )

        return self.net(feat)


class NoiseAdaptiveResidualDemucs(nn.Module):
    """
    Exact Demucs forward with continuous noise conditioning.

    At zero initialization:
        strength = 1
        gamma = beta = 0
        enhanced = speech_candidate exactly

    Enhancement is represented as learned residual subtraction:
        predicted_noise = strength * (input - speech_candidate)
        enhanced = input - predicted_noise

    The numerically stable equivalent expression is used so initialization
    reproduces the baseline without subtraction-cancellation error.
    """

    def __init__(
        self,
        demucs,
        embedding=128,
        film_scale=0.10,
        strength_range=0.15,
    ):
        super().__init__()

        self.base = demucs
        self.conditioner = SpectralNoiseConditioner(
            embedding=embedding
        )

        bottleneck = int(
            self.base.encoder[-1][0].out_channels
        )

        self.film = nn.Linear(
            embedding,
            2 * bottleneck
        )

        self.strength = nn.Linear(
            embedding,
            1
        )

        # Exact baseline behaviour at initialization.
        nn.init.zeros_(self.film.weight)
        nn.init.zeros_(self.film.bias)
        nn.init.zeros_(self.strength.weight)
        nn.init.zeros_(self.strength.bias)

        self.film_scale = float(film_scale)
        self.strength_range = float(strength_range)

    def forward(self, mix, return_aux=False):
        if mix.dim() == 2:
            mix = mix.unsqueeze(1)

        original = mix
        embedding = self.conditioner(original[:, 0])

        if self.base.normalize:
            std = (
                original
                .mean(1, keepdim=True)
                .std(-1, keepdim=True)
            )
            x = original / (self.base.floor + std)
        else:
            std = 1.0
            x = original

        length = x.shape[-1]

        x = F.pad(
            x,
            (0, self.base.valid_length(length) - length)
        )

        if self.base.resample == 2:
            x = upsample2(x)
        elif self.base.resample == 4:
            x = upsample2(upsample2(x))

        skips = []

        for encode in self.base.encoder:
            x = encode(x)
            skips.append(x)

        x = x.permute(2, 0, 1)
        x, _ = self.base.lstm(x)
        x = x.permute(1, 2, 0)

        gamma, beta = self.film(embedding).chunk(2, dim=-1)

        x = (
            x * (
                1
                + self.film_scale
                * torch.tanh(gamma)[:, :, None]
            )
            + self.film_scale
            * torch.tanh(beta)[:, :, None]
        )

        for decode in self.base.decoder:
            skip = skips.pop(-1)
            x = decode(
                x + skip[..., :x.shape[-1]]
            )

        if self.base.resample == 2:
            x = downsample2(x)
        elif self.base.resample == 4:
            x = downsample2(downsample2(x))

        speech_candidate = std * x[..., :length]

        strength = (
            1
            + self.strength_range
            * torch.tanh(self.strength(embedding))[:, :, None]
        )

        residual = original - speech_candidate
        predicted_noise = strength * residual

        # Algebraically identical to:
        # enhanced = original - predicted_noise
        #
        # This stable form exactly returns speech_candidate when
        # strength == 1 and avoids floating-point cancellation.
        enhanced = (
            speech_candidate
            + (1 - strength) * residual
        )

        if return_aux:
            return (
                enhanced,
                predicted_noise,
                strength,
                embedding,
            )

        return enhanced


model = NoiseAdaptiveResidualDemucs(base)

# Verify exact initialization against the original Demucs.
base.eval()
model.eval()

with torch.no_grad():
    probe = torch.randn(2, 1, 16000)

    expected = base(probe)
    actual = model(probe)

    difference = expected - actual

    rel = float(
        difference.norm()
        / expected.norm().clamp_min(1e-8)
    )

    max_abs = float(
        difference.abs().max()
    )

if rel > 1e-6 or max_abs > 1e-6:
    raise RuntimeError(
        "Zero-initialized wrapper does not reproduce baseline: "
        f"relative_error={rel}, max_absolute_error={max_abs}"
    )

INITIAL_CHECKPOINT = RUN / "adaptive_initial.pt"

# Replace an earlier initial checkpoint created using the unstable expression.
initial_metadata = RUN / "adaptive_initial_metadata.json"
initial_signature = sha_json({
    "implementation": IMPLEMENTATION,
    "baseline_sha": BASELINE_SHA,
    "stable_residual_expression": True,
})

reuse_initial = False

if INITIAL_CHECKPOINT.exists() and initial_metadata.exists():
    cached = json.loads(initial_metadata.read_text())

    reuse_initial = (
        cached.get("initial_signature")
        == initial_signature
    )

if not reuse_initial:
    temporary = INITIAL_CHECKPOINT.with_suffix(".tmp")

    torch.save(
        {
            key: value.detach().cpu()
            for key, value in model.state_dict().items()
        },
        temporary,
    )

    os.replace(
        temporary,
        INITIAL_CHECKPOINT
    )

    put_json(
        initial_metadata,
        {
            "initial_signature": initial_signature,
            "baseline_checkpoint": str(BASELINE_CHECKPOINT),
            "baseline_sha256": BASELINE_SHA,
            "relative_error": rel,
            "max_absolute_error": max_abs,
        },
    )

print(
    "ADAPTIVE INITIALIZATION PASS",
    "| relative baseline error:", rel,
    "| maximum absolute error:", max_abs,
    "| baseline SHA:", BASELINE_SHA,
)

ADAPTIVE INITIALIZATION PASS | relative baseline error: 0.0 | maximum absolute error: 0.0 | baseline SHA: da6762a5383c838254e82cbadb27e29b6b51872ca29c06018be25ae703b90a7c


## 5 — Balanced genuine/pseudo residual training with restartable checkpoints


In [ ]:
def neg_si_sdr_items(y,s):
    y=y-y.mean(-1,keepdim=True);s=s-s.mean(-1,keepdim=True)
    target=(y*s).sum(-1,keepdim=True)/(s.square().sum(-1,keepdim=True)+1e-8)*s
    error=y-target
    return -10*torch.log10((target.square().sum(-1)+1e-8)/(error.square().sum(-1)+1e-8))/20

def multires_stft_loss(y,s):
    total=0
    for size in (256,512,1024):
        win=torch.hann_window(size,device=y.device)
        Y=torch.stft(y,size,hop_length=size//4,window=win,return_complex=True,pad_mode="constant").abs()
        S=torch.stft(s,size,hop_length=size//4,window=win,return_complex=True,pad_mode="constant").abs()
        total+=(torch.log1p(Y)-torch.log1p(S)).abs().mean()
    return total/3

def gain_matched_targets(x,s,event_mask):
    outside=1-event_mask
    den=(s.square()*outside).sum(-1,keepdim=True)
    num=(x*s*outside).sum(-1,keepdim=True)
    fallback_den=s.square().sum(-1,keepdim=True)
    fallback_num=(x*s).sum(-1,keepdim=True)
    gain=torch.where(den>1e-5,num/den.clamp_min(1e-8),fallback_num/fallback_den.clamp_min(1e-8))
    gain=gain.clamp(.5,1.5)
    target_s=gain*s;target_n=x-target_s
    return target_s,target_n,gain

def enhancement_loss(net,x,s,event_mask):
    y,pred_n,strength,_=net(x[:,None],return_aux=True);y=y[:,0];pred_n=pred_n[:,0]
    target_s,target_n,gain=gain_matched_targets(x,s,event_mask)
    l_s=neg_si_sdr_items(y,target_s).mean()
    noise_energy=target_n.square().mean(-1);valid=noise_energy>1e-7
    l_n=neg_si_sdr_items(pred_n[valid],target_n[valid]).mean() if valid.any() else y.new_zeros(())
    scale=target_s.square().mean(-1).sqrt().clamp_min(.01)[:,None]
    l_wave=((y-target_s)/scale).abs().mean();l_stft=multires_stft_loss(y,target_s)
    outside=1-event_mask
    preserve=(((y-x).abs()/scale)*outside).sum()/outside.sum().clamp_min(1)
    loss=l_s+NOISE_LOSS_WEIGHT*l_n+STFT_WEIGHT*l_stft+WAVE_WEIGHT*l_wave+PRESERVE_WEIGHT*preserve
    return loss,dict(speech_si=l_s,noise_si=l_n,stft=l_stft,wave=l_wave,preserve=preserve,
                     strength=strength.mean()),y

class PairDataset(Dataset):
    def __init__(self,records,training=True):self.records=list(records);self.training=training;self.epoch=0
    def set_epoch(self,epoch):self.epoch=int(epoch)
    def __len__(self):return len(self.records)
    def __getitem__(self,i):
        r=self.records[i];x=read_wave_strict(r["noisy_path"]);s=read_wave_strict(r["clean_path"])
        if len(x)!=len(s):raise ValueError("Pair length mismatch")
        evs=event_dicts(r.get("events",[]));n=min(len(x),SEGMENT)
        rng=np.random.default_rng(SEED+i*1000003+self.epoch*10007)
        u=float(rng.random()) if self.training else 0.
        identity=self.training and u<IDENTITY_PROBABILITY
        random_crop=self.training and IDENTITY_PROBABILITY<=u<IDENTITY_PROBABILITY+RANDOM_CROP_PROBABILITY
        if identity:x=s.copy();evs=[]
        if random_crop or not evs:
            start=int(rng.integers(0,len(x)-n+1)) if self.training and len(x)>n else max(0,(len(x)-n)//2)
        else:
            e=evs[int(rng.integers(0,len(evs)))] if self.training else evs[0]
            a=max(0,min(len(x)-1,int(e["start"]*SR)));b=max(a+1,min(len(x),int(e["end"]*SR)))
            anchor=int(rng.integers(a,b));offset=int(rng.integers(0,n+1)) if self.training else n//2
            start=max(0,min(len(x)-n,anchor-offset))
        xx=np.pad(x[start:start+n],(0,SEGMENT-n));ss=np.pad(s[start:start+n],(0,SEGMENT-n))
        mask=np.zeros(SEGMENT,np.float32)
        for e in evs:
            a=max(0,int(e["start"]*SR)-start);b=min(n,int(e["end"]*SR)-start)
            if b>a:mask[a:b]=1
        return torch.from_numpy(xx.astype(np.float32)),torch.from_numpy(ss.astype(np.float32)),torch.from_numpy(mask)

pseudo_records=PSEUDO[PSEUDO.split=="pseudo_train"].to_dict("records")
real_records=[dict(pair_id="real_"+r["clip_id"],noisy_path=r["noisy_path"],clean_path=r["clean_path"],events=r["events"])
              for r in TRAIN_ROWS.to_dict("records")]
REAL_REPEAT=max(1,round(len(pseudo_records)/len(real_records)))
train_records=pseudo_records+real_records*REAL_REPEAT
val_records=PSEUDO[PSEUDO.split=="pseudo_val"].to_dict("records")
train_ds=PairDataset(train_records,True);val_ds=PairDataset(val_records,False)
train_dl=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=True,drop_last=True)
val_dl=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=True)
real_fraction=len(real_records)*REAL_REPEAT/len(train_records)
print("Training records",len(train_records),"| genuine update fraction",real_fraction,"| real repeat",REAL_REPEAT)
print("Crop policy",dict(event_centred=EVENT_CENTRED_PROBABILITY,random=RANDOM_CROP_PROBABILITY,identity=IDENTITY_PROBABILITY))


Training records 9219 | genuine update fraction 0.5128538887081028 | real repeat 8
Crop policy {'event_centred': 0.8, 'random': 0.15, 'identity': 0.05}


In [ ]:
history_path=RUN/"adaptive_training_history.csv"
history=pd.read_csv(history_path).to_dict("records") if history_path.exists() else []
existing=sorted(RUN.glob("adaptive_epoch_*.pt"));start_epoch=0
if existing:
    latest=existing[-1];model.load_state_dict(torch.load(latest,map_location="cpu",weights_only=True),strict=True)
    start_epoch=int(latest.stem.rsplit("_",1)[-1]);print("Resuming after epoch",start_epoch)

def flatten_rnns(net):
    for module in net.modules():
        if isinstance(module,nn.RNNBase):module.flatten_parameters()

def configure_stage(epoch):
    for p in model.parameters():p.requires_grad_(False)
    for p in model.conditioner.parameters():p.requires_grad_(True)
    for p in model.film.parameters():p.requires_grad_(True)
    for p in model.strength.parameters():p.requires_grad_(True)
    groups=[dict(params=list(model.conditioner.parameters())+list(model.film.parameters())+list(model.strength.parameters()),lr=LR_ADAPTER)]
    stage="adapter"
    if epoch>=2:
        for p in model.base.decoder.parameters():p.requires_grad_(True)
        groups.append(dict(params=model.base.decoder.parameters(),lr=LR_DECODER));stage="adapter_decoder"
    if epoch>=3:
        for p in model.base.encoder.parameters():p.requires_grad_(True)
        for p in model.base.lstm.parameters():p.requires_grad_(True)
        groups.append(dict(params=list(model.base.encoder.parameters())+list(model.base.lstm.parameters()),lr=LR_FULL));stage="full_low_lr"
    return torch.optim.AdamW(groups,weight_decay=1e-5),stage

@torch.inference_mode()
def validation_loss():
    model.eval();flatten_rnns(model);vals=[]
    for x,s,m in val_dl:
        x,s,m=x.to(DEVICE),s.to(DEVICE),m.to(DEVICE)
        with torch.amp.autocast("cuda",enabled=True):vals.append(float(enhancement_loss(model,x,s,m)[0]))
    return float(np.mean(vals))

CANDIDATES={"adaptive_initial":INITIAL_CHECKPOINT};model.to(DEVICE)
for epoch in range(start_epoch+1,FINETUNE_EPOCHS+1):
    train_ds.set_epoch(epoch);optimizer,stage=configure_stage(epoch)
    model.train();flatten_rnns(model);values=[];parts_sum=[]
    for x,s,m in tqdm(train_dl,desc=f"Noise-adaptive residual training {epoch}/{FINETUNE_EPOCHS}"):
        x,s,m=x.to(DEVICE,non_blocking=True),s.to(DEVICE,non_blocking=True),m.to(DEVICE,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda",enabled=True):loss,parts,_=enhancement_loss(model,x,s,m)
        if not torch.isfinite(loss):raise RuntimeError("Non-finite training loss")
        loss.backward();trainable=[p for p in model.parameters() if p.requires_grad]
        nn.utils.clip_grad_norm_(trainable,5.,error_if_nonfinite=True);optimizer.step()
        values.append(float(loss.detach()));parts_sum.append([float(parts[k].detach()) for k in
            ("speech_si","noise_si","stft","wave","preserve","strength")])
    val=validation_loss();means=np.mean(parts_sum,axis=0)
    rec=dict(epoch=epoch,stage=stage,train_loss=float(np.mean(values)),val_loss=val,
        speech_si=float(means[0]),noise_si=float(means[1]),stft=float(means[2]),wave=float(means[3]),
        preserve=float(means[4]),mean_strength=float(means[5]))
    history=[r for r in history if int(r["epoch"])!=epoch]+[rec];history=sorted(history,key=lambda r:int(r["epoch"]))
    print(rec);path=RUN/f"adaptive_epoch_{epoch:02d}.pt";tmp=path.with_suffix(".tmp")
    torch.save({k:v.detach().cpu() for k,v in model.state_dict().items()},tmp);os.replace(tmp,path)
    pd.DataFrame(history).to_csv(history_path,index=False);del optimizer
for p in sorted(RUN.glob("adaptive_epoch_*.pt")):CANDIDATES[p.stem]=p
model.cpu();gc.collect();torch.cuda.empty_cache();print("Candidates",list(CANDIDATES))


Noise-adaptive residual training 1/3:   0%|          | 0/2304 [00:00<?, ?it/s]

{'epoch': 1, 'stage': 'adapter', 'train_loss': -1.2451418708337263, 'val_loss': -0.9800956705585122, 'speech_si': -1.0328041814953193, 'noise_si': -0.7470169078409931, 'stft': 0.015523188420148572, 'wave': 0.09743891424416991, 'preserve': 0.012473387705742627, 'mean_strength': 0.9999921586778429}


Noise-adaptive residual training 2/3:   0%|          | 0/2304 [00:00<?, ?it/s]

{'epoch': 2, 'stage': 'adapter_decoder', 'train_loss': -1.2576906903026004, 'val_loss': -0.987050793133676, 'speech_si': -1.0339510926342983, 'noise_si': -0.7813197171247642, 'stft': 0.014941835122891057, 'wave': 0.0927191578268523, 'preserve': 0.006373201429874421, 'mean_strength': 0.9996388753255209}


Noise-adaptive residual training 3/3:   0%|          | 0/2304 [00:00<?, ?it/s]

{'epoch': 3, 'stage': 'full_low_lr', 'train_loss': -1.2673231519842778, 'val_loss': -0.9928285386413336, 'speech_si': -1.034187920145794, 'noise_si': -0.8111658067338996, 'stft': 0.014691644772104459, 'wave': 0.09054563358859014, 'preserve': 0.004253770914677564, 'mean_strength': 0.9996935526529948}
Candidates ['adaptive_initial', 'adaptive_epoch_01', 'adaptive_epoch_02', 'adaptive_epoch_03']


## 6 — Always-on inference and frozen SraVaani evaluation


In [ ]:
LAST_MODEL_CALLS=0
@torch.inference_mode()
def enhance_audio(y):
    global LAST_MODEL_CALLS
    y=np.asarray(y,np.float32);out=np.zeros(len(y),np.float64);den=np.zeros(len(y),np.float64)
    weight=np.maximum(np.hanning(CHUNK_SAMPLES),1e-3);LAST_MODEL_CALLS=0
    for start in range(0,len(y),CHUNK_HOP):
        count=min(CHUNK_SAMPLES,len(y)-start);seg=np.pad(y[start:start+count],(0,CHUNK_SAMPLES-count))
        xt=torch.from_numpy(seg)[None].to(DEVICE)
        z=model(xt[:,None])[:,0][0].float().cpu().numpy()
        if not np.isfinite(z).all():raise RuntimeError("Invalid enhancement; no fallback")
        out[start:start+count]+=z[:count]*weight[:count];den[start:start+count]+=weight[:count];LAST_MODEL_CALLS+=1
        if start+CHUNK_SAMPLES>=len(y):break
    if (den<=0).any():raise RuntimeError("Uncovered samples")
    return (out/den).astype(np.float32)

def activate(path,device=DEVICE):
    model.load_state_dict(torch.load(path,map_location="cpu",weights_only=True),strict=True)
    model.to(device).eval();flatten_rnns(model)


In [ ]:
from transformers import AutoModel
pin=json.loads(ASR_PIN.read_text()); ASR_REV=str(pin['revision'])
if pin.get('repo')!=ASR_REPO or not re.fullmatch('[0-9a-fA-F]{40}',ASR_REV):
    raise ValueError('Invalid existing SraVaani revision pin')
asr_identity=dict(repo=ASR_REPO,revision=ASR_REV,task='transcribe_return_hypotheses',
    pipeline='v29_content_filename_nograd_v1',torch=str(torch.__version__),
    transformers=importlib.metadata.version('transformers'))
ASR_SIGNATURE=sha_json(asr_identity)
token=os.environ.get('HF_TOKEN')
if not token:
    try:
        from google.colab import userdata
        token=userdata.get('HF_TOKEN')
    except Exception:pass
asr_model=AutoModel.from_pretrained(ASR_REPO,revision=ASR_REV,code_revision=ASR_REV,
    trust_remote_code=True,token=token).to(DEVICE).eval()
put_json(RUN/'asr_identity.json',asr_identity)

class FreshASR:
    def __init__(self,model):
        self.model=model;self.cache=ROOT/'track2_phase2_v28_pretrained_wer'/'asr_cache_v28'/ASR_SIGNATURE
        self.cache.mkdir(parents=True,exist_ok=True)
        self.stage=WORK/'asr_inputs';self.stage.mkdir(exist_ok=True)
    def transcribe(self,paths,force=False):
        result={}
        for cid,p in tqdm(paths.items(),desc='Content-audited SraVaani'):
            p=Path(p);h=sha_file(p);dest=self.cache/(h+'.json')
            rec=None
            if dest.exists() and not force:
                trial=json.loads(dest.read_text())
                if trial.get('wav_sha256')==h and trial.get('model_signature')==ASR_SIGNATURE:
                    rec=trial
            if rec is None:
                staged=self.stage/(h+'.wav')
                if not staged.exists() or sha_file(staged)!=h:shutil.copyfile(p,staged)
                if sha_file(staged)!=h:raise IOError('ASR input hash mismatch')
                # One audio per call makes result-to-file association unambiguous.
                with torch.inference_mode(False), torch.no_grad():
                    hyp=self.model.transcribe([str(staged)],return_hypotheses=True)
                text=hyp_text(hyp)
                rec=dict(text=text,wav_sha256=h,model_signature=ASR_SIGNATURE,
                         staged_name=staged.name,source_path=str(p))
                put_json(dest,rec)
            if not isinstance(rec.get('text'),str):raise ValueError('Invalid cached transcript')
            result[cid]=rec
        return result
ASR=FreshASR(asr_model)

def text_tokens(text):
    text=re.sub(r'<[^>]*>|\[[^\]]*\]',' ',text)
    text=unicodedata.normalize('NFC',text).lower()
    return ''.join(' ' if unicodedata.category(c).startswith('P') else c for c in text).split()

def word_counts(ref,hyp):
    # Deterministic minimum-edit alignment; ties prefer substitution, then deletion.
    a,b=text_tokens(ref),text_tokens(hyp);n,m=len(a),len(b)
    dp=np.zeros((n+1,m+1),dtype=np.int32)
    dp[:,0]=np.arange(n+1);dp[0,:]=np.arange(m+1)
    for i in range(1,n+1):
        for j in range(1,m+1):
            dp[i,j]=min(dp[i-1,j-1]+(a[i-1]!=b[j-1]),dp[i-1,j]+1,dp[i,j-1]+1)
    i,j=n,m;sub=delete=insert=hit=0
    while i or j:
        if i and j and a[i-1]==b[j-1] and dp[i,j]==dp[i-1,j-1]:hit+=1;i-=1;j-=1
        elif i and j and dp[i,j]==dp[i-1,j-1]+1:sub+=1;i-=1;j-=1
        elif i and dp[i,j]==dp[i-1,j]+1:delete+=1;i-=1
        else:insert+=1;j-=1
    return dict(words=n,substitutions=sub,deletions=delete,insertions=insert,
                errors=sub+delete+insert,hits=hit)
assert word_counts('a b','a c')['substitutions']==1
assert word_counts('a b','a')['deletions']==1
assert word_counts('a','a b')['insertions']==1

BASELINES={}
def get_reference_decodes(label,rows):
    if label in BASELINES:return BASELINES[label]
    raw={r.clip_id:Path(r.noisy_path) for r in rows.itertuples()}
    raw_h=ASR.transcribe(raw)
    clean_h=ASR.transcribe({r.clip_id:Path(r.clean_path) for r in rows.itertuples()}) if 'clean_path' in rows else None
    BASELINES[label]=(raw_h,clean_h)
    return raw_h,clean_h
get_reference_decodes('synthetic_dev',SYN_DEV)
get_reference_decodes('natural_dev',NAT_DEV)
# Independent uncached re-decode of one noisy/clean pair.
first=SYN_DEV.iloc[0]
check=ASR.transcribe({'noisy':Path(first.noisy_path),'clean':Path(first.clean_path)},force=True)
put_json(RUN/'uncached_pair_check.json',check)
print('First pair fresh decode:',{k:v['text'] for k,v in check.items()})


In [ ]:
def process_rows(rows,directory,checkpoint_sha):
    directory.mkdir(parents=True,exist_ok=True);paths={};audit=[]
    for r in tqdm(rows.to_dict('records'),desc='Always-on enhancement'):
        cid=r['clip_id'];x=read_wave_strict(r['noisy_path'])
        p=directory/(cid+'.wav');rec_path=directory/(cid+'.audit.json')
        key=sha_json(dict(input=hashlib.sha256(x.tobytes()).hexdigest(),checkpoint=checkpoint_sha,impl=IMPLEMENTATION))
        rec=None
        if p.exists() and rec_path.exists():
            trial=json.loads(rec_path.read_text())
            if trial.get('key')==key and trial.get('wav_sha256')==wav_audit(p,len(x)):rec=trial
        if rec is None:
            z=enhance_audio(x);write_pcm(p,z)
            rec=dict(clip_id=cid,key=key,model_calls=LAST_MODEL_CALLS,wav_sha256=wav_audit(p,len(x)),
                unchanged_pcm=bool(np.array_equal(canonical_pcm(x),canonical_pcm(z))),
                clipped_samples=int(np.count_nonzero((z<-1)|(z>32767/32768))),
                change_rms=float(np.sqrt(np.mean((pcm_roundtrip(z)-pcm_roundtrip(x))**2))))
            put_json(rec_path,rec)
        paths[cid]=p;audit.append(rec)
    return paths,audit

def evaluate(label,rows,paths,candidate):
    raw,clean=get_reference_decodes(label,rows)
    enh=ASR.transcribe(paths);details=[];skipped=0
    for r in rows.to_dict('records'):
        cid=r['clip_id'];ref=r['transcript']
        if not text_tokens(ref):skipped+=1;continue
        record=dict(clip_id=cid,reference=ref,raw_text=raw[cid]['text'],enhanced_text=enh[cid]['text'],
            raw_sha=raw[cid]['wav_sha256'],enhanced_sha=enh[cid]['wav_sha256'],
            same_transcript=text_tokens(raw[cid]['text'])==text_tokens(enh[cid]['text']))
        for role,h in [('raw',raw),('enhanced',enh),('clean',clean)]:
            if h is not None:
                record.update({role+'_'+k:v for k,v in word_counts(ref,h[cid]['text']).items()})
                if role=='clean':record['clean_text']=h[cid]['text'];record['clean_sha']=h[cid]['wav_sha256']
        if 'clean_path' in r:
            s=read_wave_strict(r['clean_path']);x=read_wave_strict(r['noisy_path']);z=read_wave_strict(paths[cid])
            record['noisy_si_sdr']=si_sdr(x,s);record['enhanced_si_sdr']=si_sdr(z,s)
        record['error_increase']=record['enhanced_errors']-record['raw_errors']
        details.append(record)
    if not details:raise ValueError('No usable reference transcripts')
    frame=pd.DataFrame(details).sort_values('error_increase',ascending=False)
    frame.to_csv(RUN/(candidate+'_'+label+'_details.csv'),index=False)
    words=int(frame.raw_words.sum());summary=dict(candidate=candidate,subset=label,clips=len(frame),skipped=skipped,reference_words=words,raw_errors=int(frame.raw_errors.sum()),enhanced_errors=int(frame.enhanced_errors.sum()),
        raw_wer_percent=100*frame.raw_errors.sum()/words,
        enhanced_wer_percent=100*frame.enhanced_errors.sum()/words,
        identical_transcript_fraction=float(frame.same_transcript.mean()))
    summary['delta_wer_points']=summary['raw_wer_percent']-summary['enhanced_wer_percent']
    for role in ['raw','enhanced']:
        for typ in ['substitutions','deletions','insertions']:summary[role+'_'+typ]=int(frame[role+'_'+typ].sum())
    if clean is not None:
        summary['clean_wer_percent']=100*frame.clean_errors.sum()/words
        summary['enhanced_si_sdr']=float(frame.enhanced_si_sdr.mean())
        summary['noisy_si_sdr']=float(frame.noisy_si_sdr.mean())
    print(summary);return summary



In [ ]:
summaries=[];selection=[]
for candidate,checkpoint in CANDIDATES.items():
    asr_model.to("cpu");torch.cuda.empty_cache();activate(checkpoint);checkpoint_sha=sha_file(checkpoint)
    syn_paths,syn_audit=process_rows(SYN_DEV,WORK/candidate/"synthetic_dev",checkpoint_sha)
    nat_paths,nat_audit=process_rows(NAT_DEV,WORK/candidate/"natural_dev",checkpoint_sha)
    model.to("cpu");torch.cuda.empty_cache();asr_model.to(DEVICE).eval()
    a=evaluate("synthetic_dev",SYN_DEV,syn_paths,candidate);b=evaluate("natural_dev",NAT_DEV,nat_paths,candidate)
    summaries.extend([a,b]);words=a["reference_words"]+b["reference_words"]
    dwer=100*((a["raw_errors"]+b["raw_errors"])-(a["enhanced_errors"]+b["enhanced_errors"]))/words
    proxy=a["enhanced_si_sdr"]+dwer
    unchanged=any(r["unchanged_pcm"] for r in syn_audit+nat_audit)
    selection.append(dict(candidate=candidate,synthetic_si_sdr=a["enhanced_si_sdr"],pooled_delta_wer_points=dwer,
                          combined_proxy=proxy,unchanged=unchanged))
    pd.DataFrame(syn_audit+nat_audit).to_csv(RUN/(candidate+"_processing_audit.csv"),index=False)
    put_json(RUN/"development_metrics.json",summaries)
eligible_sel=[r for r in selection if not r["unchanged"]]
if not eligible_sel:raise RuntimeError("No genuinely processing checkpoint")
BEST=max(eligible_sel,key=lambda r:r["combined_proxy"])["candidate"]
SELECTED=CANDIDATES[BEST];SELECTED_SHA=sha_file(SELECTED)
put_json(RUN/"selected_model.json",dict(candidate=BEST,checkpoint=str(SELECTED),sha256=SELECTED_SHA,scores=selection,
         criterion="synthetic SI-SDR plus pooled WER improvement points"))
display(pd.DataFrame(selection).sort_values("combined_proxy",ascending=False));print("Selected:",BEST)

# Monitoring is reported once and never used to reselect among candidates.
asr_model.to("cpu");torch.cuda.empty_cache();activate(SELECTED)
sp,sa=process_rows(SYN_MON,WORK/BEST/"synthetic_monitor",SELECTED_SHA)
npth,na=process_rows(NAT_MON,WORK/BEST/"natural_monitor",SELECTED_SHA)
model.to("cpu");torch.cuda.empty_cache();asr_model.to(DEVICE).eval()
monitor=[evaluate("synthetic_monitor",SYN_MON,sp,BEST),evaluate("natural_monitor",NAT_MON,npth,BEST)]
put_json(RUN/"monitor_metrics.json",monitor)


## 7 — Generate the selected Phase-2 submission ZIP


In [ ]:
if RUN_PHASE2:
    if not CONFIRM_RULES_ALLOW_SOURCE_DERIVED_TRAINING:raise RuntimeError("Rules confirmation required")
    ids=sorted(TRANSCRIPT_MAP)
    if len(ids)!=EXPECTED_PHASE2_CLIPS:raise ValueError("Unexpected Phase-2 clip count")
    source=StrictWaveSource(SOURCE_AUDIO,ids)
    asr_model.to("cpu");torch.cuda.empty_cache();activate(SELECTED)
    wavs=WORK/"phase2"/SELECTED_SHA[:12];wavs.mkdir(parents=True,exist_ok=True)
    paths={};frames={};audits=[]
    for cid in tqdm(ids,desc="All-clip always-on noise-adaptive enhancement"):
        x=source.load(cid);z=enhance_audio(x);p=wavs/(cid+".wav");write_pcm(p,z)
        audits.append(dict(clip_id=cid,input_sha256=hashlib.sha256(x.tobytes()).hexdigest(),
            wav_sha256=wav_audit(p,len(x)),checkpoint_sha256=SELECTED_SHA,model_calls=LAST_MODEL_CALLS,
            unchanged_pcm=bool(np.array_equal(canonical_pcm(x),canonical_pcm(z))),
            clipped_samples=int(np.count_nonzero((z<-1)|(z>32767/32768)))))
        paths[cid]=p;frames[cid]=len(x)
    source.close();pd.DataFrame(audits).to_csv(RUN/"phase2_processing_audit.csv",index=False)
    if any(r["unchanged_pcm"] for r in audits):raise RuntimeError("Unchanged output found; no bypass permitted")
    model.to("cpu");torch.cuda.empty_cache();asr_model.to(DEVICE).eval();transcripts=ASR.transcribe(paths)
    ZIP=RUN/("submission_Track2_PHASE2_V34_NoiseAdaptive_"+BEST+"_"+SELECTED_SHA[:10]+".zip")
    package=pack_submission(paths,transcripts,frames,ZIP)
    put_json(RUN/"submission_manifest.json",dict(version="v34_noise_adaptive_residual_demucs",candidate=BEST,
        checkpoint_sha256=SELECTED_SHA,baseline_checkpoint_sha256=BASELINE_SHA,all_clips_enhanced=True,
        routing=False,dictionary_inference=False,classifier_routing=False,beats_in_inference=False,
        tts_in_inference=False,continuous_noise_conditioning=True,residual_subtraction=True,
        source_derived_training_permission_confirmed=True,package=package))
    print("ZIP ready for manual upload:",ZIP)
else:print("RUN_PHASE2=False: metrics generated; no submission ZIP created.")


## Outputs

All V34 artifacts are written under:

`/content/drive/MyDrive/PHD/Datathon/track2_phase2_v34_noise_adaptive_residual_demucs/run_v1`

The model begins as an exact numerical reproduction of the current baseline.
The baseline candidate therefore remains selectable if adaptive training does
not improve the untouched synthetic selection set plus natural-development WER.
